CSV data cleanup - Seriemas terrain points
=
This notebook cleans the point CSV files of the Seriemas project (one per park/area, e.g. `..._MDS_PARQUE 04_R01.csv`). For each file it keeps only the X, Y and elevation columns and writes a headerless copy that starts with a `sep=,` line (the line that tells Excel which separator the file uses).

**Workflow overview:**
1. List the `.csv` files in the input folder (top level only, subfolders aren't searched).
2. Read each file with Polars and keep `X`, `Y`, `ELEVATION` (renamed to `Z`).
3. Write `<name>-cleaned.csv` into the `cleaned` subfolder without a header, then prepend `sep=,`.

**Input prerequisites:**
- **Folder:** the CSVs are in `C:\Users\Usuario\Desktop\seriemas`. Every file ending in `.csv` there is processed (the `.csv` check is case-sensitive, so `.CSV` files are skipped).
- **Format:** comma-separated, with a header row.
- **Columns:** `X`, `Y` and `ELEVATION`, with these exact names (case-sensitive). Other columns are dropped. A file without one of them stops the loop with an error.

Cleaning every CSV in the folder
=
- Creates `output_dir` (`...\seriemas\cleaned`) if needed. It's a subfolder, so the cleaned files aren't picked up again on the next run (only the top level is listed).
- For each `.csv`: `scan_csv` → `select(['X', 'Y', 'ELEVATION'])` → rename `ELEVATION` to `Z` → `collect().write_csv(..., include_header=False)`.
- The rename has no visible effect, because the header isn't written.
- The file is then reopened in `r+` mode, read into memory and rewritten with `sep=,` as the first line. So the output is `sep=,` followed by `X,Y,Z` values, one point per line, with no header.
- Existing `-cleaned.csv` files are overwritten.

In [23]:
import polars as pl
import os

directory = r"C:\Users\Usuario\Desktop\seriemas"
output_dir = r"C:\Users\Usuario\Desktop\seriemas\cleaned"
os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(directory):
    if file.endswith(".csv"):
        print(f"Processing: {file}")
        file_path = os.path.join(directory, file)
        output_path = os.path.join(output_dir, f"{file[:-4]}-cleaned.csv")
        print(f"Output: {output_path}")

        df = pl.scan_csv(file_path)
        df = df.select(['X', 'Y', 'ELEVATION'])
        df = df.rename({'ELEVATION': 'Z'})

        # Write the sep=, line and then the data without header
        # Since we can't append with sink_csv, we'll use write_csv with a custom approach
        df.collect().write_csv(output_path, include_header=False)

        # Then prepend the sep=, line
        with open(output_path, 'r+') as f:
            content = f.read()
            f.seek(0, 0)
            f.write('sep=,\n' + content)

Processing: 20260319_UFV_CONSORCIO SERIEMAS_MDS_PARQUE 04_R01.csv
Output: C:\Users\Usuario\Desktop\seriemas\cleaned\20260319_UFV_CONSORCIO SERIEMAS_MDS_PARQUE 04_R01-cleaned.csv
Processing: 20260319_UFV_CONSORCIO SERIEMAS_MDS_PARQUE 05_R01.csv
Output: C:\Users\Usuario\Desktop\seriemas\cleaned\20260319_UFV_CONSORCIO SERIEMAS_MDS_PARQUE 05_R01-cleaned.csv
